# BCC: certified output-head edit capacity

**Question:** At a 1% prediction-risk limit, how many weights may be pruned,
sign-flipped, or reduced in magnitude in any permitted combination?

Choose **Runtime → Change runtime type → A100 GPU**, with high RAM. Run Steps
1–4 in order. Start with **SmolLM2-1.7B**; afterward choose **OLMoE-1B-7B** in Step 3.
Each model is a separate resumable run. No separate code ZIP is needed.

The fixed experiment uses 1,024-token contexts, 32-token greedy continuations,
8,192 independent prefixes, two cache formats, three selection methods and
nine common edit sizes. All outcomes and matched controls are retained.

A session runs for up to 9.5 compute hours before checkpoint export. Completion
may require another session. Resume with the same GPU model and pinned runtime;
the protocol is never reduced to fit a session. The notebook saves completed
chunks to Drive and provides a small review ZIP for analysis.

Software validation uses tiny CPU models. The public-model GPU experiments
have not been run here and no scientific outcome is prefilled.


## 1. Load the verified experiment source

In [ ]:
import base64,hashlib,io,zipfile
from pathlib import Path
BASE=Path('/content/BCC_edit_capacity_v1')
SOURCE=BASE/'source_c1ae48a8c5c8'
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='2036047af17accb4b205d505a566f7015875f51d68bb8f685de01653d3102f61','Embedded source checksum failed'
SOURCE.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for name in z.namelist():
        assert not Path(name).is_absolute() and '..' not in Path(name).parts
        p=SOURCE/name;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(z.read(name))
print('Verified experiment source:',SOURCE)


## 2. Mount Drive and prepare the environment

This setup repairs partially created virtual environments and works without
`ensurepip`. It installs pinned packages, runs the mathematical tests, and
checks available GPU memory, host RAM and disk. It leaves the notebook kernel's
packages unchanged.


In [ ]:
# Embedded source is written by the preceding cell.
import os, sys, subprocess, json, hashlib, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_BASE=Path('/content/drive/MyDrive/BCC_edit_capacity')
DRIVE_BASE.mkdir(parents=True,exist_ok=True)
BASE=Path('/content/BCC_edit_capacity_v1')
VENV=BASE/'venv'
PYTHON=VENV/'bin/python'
subprocess.run(['nvidia-smi'],check=True)

# Colab's distribution Python may have venv but no ensurepip. Repair partial
# environments as well as creating new ones, without changing the system Python.
wheel=SOURCE/'vendor/pip-25.2-py3-none-any.whl'
expected='6d67a2b4e7f14d8b31b8b52648866fa717f45a1eb70e83002f4331d07e953717'
if not wheel.is_file() or hashlib.sha256(wheel.read_bytes()).hexdigest()!=expected:
    raise RuntimeError('Bundled pip is missing or damaged. Run Step 1 of the corrected notebook first.')
def bootstrap_step(command):
    result=subprocess.run([str(x) for x in command],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    if result.stdout:print(result.stdout,flush=True)
    if result.returncode:
        raise RuntimeError(f'Environment bootstrap failed (exit {result.returncode}). See the output above.')
bootstrap_step([sys.executable,'-I','-m','venv','--without-pip',VENV])
bootstrap=(
    'import runpy,sys; wheel=sys.argv[1]; sys.path.insert(0,wheel); '
    'sys.argv=["pip","install","--no-index","--no-deps",'
    '"--disable-pip-version-check",wheel]; runpy.run_module("pip",run_name="__main__")'
)
bootstrap_step([PYTHON,'-I','-c',bootstrap,wheel])
bootstrap_step([PYTHON,'-I','-m','pip','--version'])
subprocess.run([str(PYTHON),'-m','pip','install','-r',str(SOURCE/'requirements.txt')],check=True)
subprocess.run([str(PYTHON),'-m','unittest','test_capacity','-v'],cwd=SOURCE,check=True)
check="import torch,psutil; print('GPU:',torch.cuda.get_device_name() if torch.cuda.is_available() else 'none'); print('Free host RAM GiB:',round(psutil.virtual_memory().available/2**30,1)); assert torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory>=35*2**30, 'Choose A100 40/80GB or equivalent'; assert psutil.virtual_memory().available>=24*2**30, 'Choose a high-RAM runtime (24 GiB free required)'"
subprocess.run([str(PYTHON),'-c',check],check=True)
if shutil.disk_usage('/content').free<35*2**30:
    raise RuntimeError('At least 35 GiB free local disk is required.')
print('Setup complete. The next cell starts or resumes the frozen experiment.')


## 3. Start or resume a model

Default: `MODEL_KEY = 'smollm2_17b'`. For the second experiment select
`olmoe_1b7b`. Automatic resume reuses the newest compatible run, including a
completed run, so rerunning this cell does not repeat completed work.
To select a specific checkpoint, paste its Drive folder name into
`RESUME_RUN_ID`. These are new edit-capacity runs; older bit-allocation or
single-family experiment checkpoints use different protocols.

After a reset, run Steps 1–3 again. The immutable Drive checkpoint restores
completed chunks, families and seeds. Public model weights are re-downloaded
by their pinned revision if absent. A tiny integer-kernel check runs before
expensive model work.


In [ ]:
import datetime, hashlib, json, os, queue, subprocess, sys, threading, time
from pathlib import Path

MODEL_KEY='smollm2_17b' #@param ['smollm2_17b','olmoe_1b7b']
SESSION_HOURS=9.5 #@param {type:'number'}
RESUME_RUN_ID='' #@param {type:'string'}
AUTO_RESUME=True #@param {type:'boolean'}

if not 0.1<=SESSION_HOURS<=20:raise ValueError('Choose a session budget from 0.1 to 20 hours.')
sys.path.insert(0,str(SOURCE))
from checkpoint import restore

def object_json(folder,manifest,name):
    entry=manifest['files'][name];p=folder/'checkpoint/objects'/entry['sha256'];b=p.read_bytes()
    if len(b)!=entry['bytes'] or hashlib.sha256(b).hexdigest()!=entry['sha256']:raise RuntimeError('Corrupt saved metadata: '+name)
    return json.loads(b)

local_source={str(p.relative_to(SOURCE)):hashlib.sha256(p.read_bytes()).hexdigest()
              for p in sorted(SOURCE.rglob('*')) if p.is_file() and '__pycache__' not in p.parts and p.suffix in ('.py','.md','.txt','.json','.whl')}
chosen=None
if RESUME_RUN_ID:
    if Path(RESUME_RUN_ID).name!=RESUME_RUN_ID:raise ValueError('Enter the run folder name only.')
    chosen=DRIVE_BASE/RESUME_RUN_ID
    if not (chosen/'checkpoint/LATEST.json').is_file():raise FileNotFoundError(f'Checkpoint manifest missing: {chosen}')
elif AUTO_RESUME:
    candidates=sorted(DRIVE_BASE.glob(f'capacity_{MODEL_KEY}_*/checkpoint/LATEST.json'),key=lambda p:p.stat().st_mtime,reverse=True)
    for latest in candidates:
        folder=latest.parent.parent;manifest=json.loads(latest.read_text())
        protocol=object_json(folder,manifest,'protocol.json');status=object_json(folder,manifest,'status.json')
        if protocol['source']==local_source:chosen=folder;break
if chosen:
    RUN_ID=chosen.name;manifest=json.loads((chosen/'checkpoint/LATEST.json').read_text())
    protocol=object_json(chosen,manifest,'protocol.json');status=object_json(chosen,manifest,'status.json')
    if protocol['config']['model_key']!=MODEL_KEY or protocol['config']['fixture']:raise ValueError('Resume model/profile mismatch')
    if protocol['source']!=local_source:raise ValueError('Resume requires the notebook with the original frozen scientific source')
    JOB=BASE/'jobs'/RUN_ID
    if not JOB.exists() or not any(JOB.iterdir()):restore(chosen/'checkpoint',JOB)
    print('Resuming:',RUN_ID,'— completed chunks are reused.')
else:
    RUN_ID=f'capacity_{MODEL_KEY}_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')
    JOB=BASE/'jobs'/RUN_ID;chosen=DRIVE_BASE/RUN_ID
    status={'execution_status':'NEW'}
    print('Starting:',RUN_ID)
chosen.mkdir(parents=True,exist_ok=True)
print('Drive folder:',chosen)
if status['execution_status']=='COMPLETE':
    print('This run is already complete. Use Step 4 to retrieve its report and review ZIP.')
else:
    command=[str(PYTHON),'-u',str(SOURCE/'experiment.py'),'--model',MODEL_KEY,'--run',str(JOB),'--drive',str(chosen),'--hours',str(SESSION_HOURS)]
    worker_env=os.environ.copy();worker_env['MPLBACKEND']='Agg';worker_env['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
    proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=worker_env)
    output=queue.Queue()
    def reader():
        for line in proc.stdout:output.put(line)
    threading.Thread(target=reader,daemon=True).start();started=time.monotonic();last=started
    try:
        while proc.poll() is None or not output.empty():
            try:print(output.get(timeout=1),end='')
            except queue.Empty:pass
            if time.monotonic()-last>=30:
                print(f'[session {(time.monotonic()-started)/3600:.2f} h; run {RUN_ID}]');last=time.monotonic()
    except KeyboardInterrupt:
        import signal
        proc.send_signal(signal.SIGINT)
        print('Requested checkpoint export. If runtime stops, resume from the verified Drive objects.')
        try:proc.wait(timeout=60)
        except subprocess.TimeoutExpired:print('Export is still running; retain the runtime if possible.')
    print('Controller exit:',proc.poll())
    if (JOB/'status.json').is_file():print((JOB/'status.json').read_text())
print('Run Step 4 to display the report and download the review ZIP, including after an incomplete run.')


## 4. Display the report and download the review ZIP

This cell works in a fresh runtime without Steps 1–3. Leave recovery on `auto`
or specify a model/run folder. If the final review ZIP is missing, it recovers
a review from the latest verified Drive checkpoint. Return the **review ZIP**
for analysis; the larger checkpoint stays in Drive.

If the browser blocks downloading, use the printed Drive path or Colab's
Files panel. An incomplete run's archive remains useful and can be resumed.


In [ ]:
# Standalone: also works after Colab resets, without running setup again.
import hashlib, json, shutil, zipfile
from pathlib import Path
from google.colab import drive, files
from IPython.display import Markdown, display

RECOVER_MODEL='auto' #@param ['auto','smollm2_17b','olmoe_1b7b']
RECOVER_RUN_ID='' #@param {type:'string'}
DOWNLOAD_REVIEW=True #@param {type:'boolean'}
drive.mount('/content/drive')
base=Path('/content/drive/MyDrive/BCC_edit_capacity')
if RECOVER_RUN_ID:
    if Path(RECOVER_RUN_ID).name!=RECOVER_RUN_ID:raise ValueError('Use the run folder name only.')
    folder=base/RECOVER_RUN_ID
elif globals().get('RUN_ID','').startswith('capacity_') and (RECOVER_MODEL=='auto' or RUN_ID.startswith(f'capacity_{RECOVER_MODEL}_')):
    folder=base/RUN_ID
else:
    model_pattern='*' if RECOVER_MODEL=='auto' else RECOVER_MODEL
    candidates=list(base.glob(f'capacity_{model_pattern}_*/checkpoint/LATEST.json'))
    if not candidates:raise FileNotFoundError(f'No {RECOVER_MODEL} checkpoint in {base}. Check the model and Drive mount.')
    folder=max(candidates,key=lambda p:p.stat().st_mtime).parent.parent
latest=folder/'checkpoint/LATEST.json';archive=folder/'BCC_capacity_review.zip'
local=Path('/content')/(folder.name+'_review.zip')
if latest.is_file() and (not archive.is_file() or latest.stat().st_mtime>archive.stat().st_mtime):
    manifest=json.loads(latest.read_text());selected={}
    with zipfile.ZipFile(local,'w',zipfile.ZIP_DEFLATED,compresslevel=3) as z:
        for name,entry in manifest['files'].items():
            p=Path(name)
            if p.is_absolute() or '..' in p.parts or '\\' in name:raise RuntimeError('Unsafe checkpoint name')
            if 'features' in p.parts or 'inputs' in p.parts or p.name in ['head.npz','head_original_rtn.npz','head_gptq_candidate.npz']:continue
            source=folder/'checkpoint/objects'/entry['sha256'];b=source.read_bytes()
            if len(b)!=entry['bytes'] or hashlib.sha256(b).hexdigest()!=entry['sha256']:raise RuntimeError('Corrupt checkpoint object: '+name)
            z.writestr(name,b);selected[name]=entry
        z.writestr('MANIFEST.json',json.dumps(selected,sort_keys=True))
    recovered=folder/'BCC_capacity_review_recovered.zip';shutil.copy2(local,recovered)
    if hashlib.sha256(local.read_bytes()).digest()!=hashlib.sha256(recovered.read_bytes()).digest():raise RuntimeError('Recovered Drive copy mismatch')
    archive=recovered
elif archive.is_file():shutil.copy2(archive,local)
elif globals().get('JOB') and (Path(JOB)/'exports/BCC_capacity_review.zip').is_file():
    archive=Path(JOB)/'exports/BCC_capacity_review.zip';shutil.copy2(archive,local)
    print('Using the verified local export; the Drive copy was unavailable.')
else:raise FileNotFoundError(f'No review ZIP or resumable checkpoint manifest in {folder}. Inspect the run-cell error.')
with zipfile.ZipFile(local) as z:
    if z.testzip():raise RuntimeError('Corrupt ZIP')
    manifest=json.loads(z.read('MANIFEST.json'))
    if len(z.namelist())!=len(set(z.namelist())) or set(z.namelist())!=set(manifest)|{'MANIFEST.json'}:raise RuntimeError('Incomplete ZIP manifest')
    for name,entry in manifest.items():
        b=z.read(name)
        if len(b)!=entry['bytes'] or hashlib.sha256(b).hexdigest()!=entry['sha256']:raise RuntimeError('Archive checksum mismatch: '+name)
    print(z.read('status.json').decode())
    if 'REPORT.md' in z.namelist():display(Markdown(z.read('REPORT.md').decode()))
    else:print('Numerical report not ready yet. Resume this same run to finish the fixed protocol.')
print('Verified review source:',archive)
print('Resumable checkpoint:',folder/'checkpoint')
print('Return this ZIP:',local.name)
print('Bytes:',local.stat().st_size,'SHA256:',hashlib.sha256(local.read_bytes()).hexdigest())
if DOWNLOAD_REVIEW:files.download(str(local))


## Reading the result

* **`capacities.csv`**: largest certified size on the common grid, individually
  and jointly across both cache settings.
* **`capacity_curve.csv` / `joint_curve.csv`**: every size, risk bound, quality
  gate, pruning/sign-flip result and conditional sign-prior credit.
* **`paired_comparisons.csv`**: BCC versus both controls at every matched size.
* **`quality.csv` / `timings.csv`**: measured quality changes and computation cost.
* **`REPORT.md` / `capacity_curve.pdf`**: readable report and plot.

The main risk event is that some permitted combination changes at least one
of the 32 declared continuation tokens. Each cache setting uses its own
unedited predictions. A shared certificate covers both cache settings.
Full-pruning and full-flip outcomes are reported separately.

The 816-entry confidence ledger covers both planned models and the complete
comparison grid at joint confidence at least 95%. Selecting the largest
passing grid point is included. A passing maximum grid point establishes at
least that capacity; it does not establish the largest possible mask.

The BCC connection is the family mass and independently measured transfer
risk. Conditional sign-code credit, head fraction and packed cache savings
are reported explicitly. This experiment does not calculate an unconditional
complete-checkpoint risk bound. The complete protocol is embedded as
`PROTOCOL.md` and included in every review ZIP.
